<a href="https://colab.research.google.com/github/prabhavism/AgriVision-AI/blob/main/eegimagenet/eegimagenet_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EEG-ImageNet pipeline (Prabhav, Member B), free Colab, CPU only

**Goal for Days 1 to 3:** download, filter, downsample, extract features, save to Drive, run SVM / LDA / Random Forest baselines.

Run top to bottom. Cells marked **EDIT** depend on the file format inside the dataset zip, which you only see after running Step 2.

Rules: save to Drive after every step, keep features not raw data, `n_jobs=2` (Colab has 2 cores).

## Step 0: Setup and config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, time, zipfile
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive/brain_decoding'
FEAT_DIR = f'{BASE}/features'
RES_DIR = f'{BASE}/results'
os.makedirs(FEAT_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)

FS_IN, FS_OUT = 1000, 128      # raw 1000 Hz -> 128 Hz
BAND = (1, 40)                 # band-pass in Hz (agree this with Rohan, KaraOne uses the same)
N_PARTICIPANTS = 4             # start small; raise if time allows
MAX_TRAIN = 20000              # cap training samples per LOSO fold (keeps RF fast on Colab)
SEED = 0

## Step 1: Download (about 14.4 GB zip, takes a while, run once)

In [ ]:
from huggingface_hub import hf_hub_download

zip_path = hf_hub_download(repo_id='PromiseZ5Q2SQ/EEG-ImageNet',
                           filename='EEG-ImageNet.zip',
                           repo_type='dataset',
                           local_dir='/content/raw')
z = zipfile.ZipFile(zip_path)
names = [n for n in z.namelist() if not n.endswith('/')]
print(len(names), 'files in zip')
print(*names[:40], sep='\n')

## Step 2: Extract a few participants and look inside  (**EDIT**)

Read the file listing above, then set `wanted` to the files of the first `N_PARTICIPANTS` participants. Extract only those, not the whole zip, to save disk.

In [ ]:
# EDIT: choose the per-participant files from the listing above
wanted = [n for n in names if n.lower().endswith(('.pth', '.pt', '.pkl', '.npy', '.mat'))][:N_PARTICIPANTS]
print(wanted)
z.extractall('/content/raw/extracted', members=wanted)
files = [f'/content/raw/extracted/{n}' for n in wanted]

import torch, pickle
def peek(path):
    obj = torch.load(path, weights_only=False) if path.endswith(('.pth', '.pt')) else pickle.load(open(path, 'rb'))
    print(type(obj))
    if isinstance(obj, dict):
        print('keys:', list(obj.keys())[:10])
    first = (obj['dataset'] if isinstance(obj, dict) and 'dataset' in obj else obj)
    item = first[0] if hasattr(first, '__getitem__') else None
    print('first item type:', type(item))
    if isinstance(item, dict):
        for k, v in item.items():
            print(' ', k, getattr(v, 'shape', v if not hasattr(v, '__len__') else len(v)))
peek(files[0])

## Step 3: Loader  (**EDIT**)

This is a best guess at the structure (a list of samples with `eeg_data` of shape 62 x 500 and a `label`). **Change the keys to match what `peek()` printed.** Must return `eeg` (n, 62, 500) float32 and `y` (n,) labels.

In [ ]:
def load_participant(path):
    obj = torch.load(path, weights_only=False)
    samples = obj['dataset'] if isinstance(obj, dict) and 'dataset' in obj else obj
    eeg = np.stack([np.asarray(s['eeg_data'], dtype=np.float32) for s in samples])   # EDIT key if needed
    y = np.array([s['label'] for s in samples])                                       # EDIT key if needed
    return eeg, y

eeg, y = load_participant(files[0])
print(eeg.shape, y.shape, len(set(y)), 'classes')

In [ ]:
for n in wanted:
    print(f'{z.getinfo(n).file_size/1e9:6.2f} GB  {n}')

In [ ]:
import gc, torch

def load_participant_lean(path, chunk=500):
    try:
        obj = torch.load(path, weights_only=False, mmap=True)   # avoids reading everything at once
    except Exception:
        obj = torch.load(path, weights_only=False)
    samples = obj['dataset'] if isinstance(obj, dict) and 'dataset' in obj else obj
    xs, ys = [], []
    for i in range(0, len(samples), chunk):
        part = samples[i:i+chunk]
        batch = np.stack([np.asarray(s['eeg_data'], dtype=np.float32) for s in part])  # EDIT key if needed
        xs.append(preprocess(batch))             # 1000 Hz -> 128 Hz right away, about 8x smaller
        ys.extend(s['label'] for s in part)      # EDIT key if needed
        del batch
    del obj, samples; gc.collect()
    return np.concatenate(xs), np.array(ys)

In [ ]:
os.remove(f'{FEAT_DIR}/eegimagenet_P00.npz')   # it mixes 8 participants under one id

wanted = [n for n in names if n.endswith('.pth')
          and '__MACOSX' not in n and not os.path.basename(n).startswith('._')]
print(wanted)
files = [f'/content/raw/extracted/{n}' for n in wanted]

In [ ]:
def load_file_lean(path, chunk=500):
    obj = torch.load(path, weights_only=False)
    samples = obj['dataset'] if isinstance(obj, dict) and 'dataset' in obj else obj
    keys = list(samples[0].keys())
    print('sample keys:', keys)
    SUBJ = next((k for k in ('subject', 'subject_id', 'sub', 'participant') if k in keys), None)
    assert SUBJ, 'no subject key found, paste the keys printed above'
    GRAN = 'granularity' if 'granularity' in keys else None
    xs, ys, subj, gran = [], [], [], []
    for i in range(0, len(samples), chunk):
        part = samples[i:i+chunk]
        batch = np.stack([np.asarray(s['eeg_data'], dtype=np.float32) for s in part])
        xs.append(preprocess(batch)); del batch
        ys.extend(s['label'] for s in part)
        subj.extend(s[SUBJ] for s in part)
        if GRAN: gran.extend(s[GRAN] for s in part)
    del obj, samples; gc.collect()
    return (np.concatenate(xs), np.array(ys), np.array(subj),
            np.array(gran) if GRAN else None)

## Step 4: Preprocess and features

Band-pass 1 to 40 Hz, downsample 1000 -> 128 Hz, drop epochs with extreme amplitude (simple outlier rule, no ICA), z-score per channel within each participant, then 7 features per channel: log band power in delta, theta, alpha, beta, gamma, plus variance and line length (62 channels x 7 = 434 features).

In [ ]:
import gc, os, time, torch
from scipy.signal import butter, sosfiltfilt, resample_poly

# remove the bad output from the earlier run (it mixed 8 participants under one id)
bad = f'{FEAT_DIR}/eegimagenet_P00.npz'
if os.path.exists(bad): os.remove(bad)

sos = butter(4, BAND, btype='bandpass', fs=FS_IN, output='sos')
BANDS = {'delta': (1, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 40)}

def preprocess(eeg):
    x = sosfiltfilt(sos, eeg, axis=-1)
    x = resample_poly(x, 16, 125, axis=-1)            # 1000 Hz -> 128 Hz
    x = x - x.mean(axis=-1, keepdims=True)
    return x.astype(np.float32)

def keep_mask(x, k=5):
    s = np.abs(x).max(axis=(1, 2))
    med = np.median(s); mad = 1.4826 * np.median(np.abs(s - med))
    return s < med + k * mad

def zscore_channels(x):
    mu = x.mean(axis=(0, 2), keepdims=True); sd = x.std(axis=(0, 2), keepdims=True) + 1e-8
    return ((x - mu) / sd).astype(np.float32)

def features(x):
    n, ch, T = x.shape
    P = np.abs(np.fft.rfft(x * np.hanning(T), axis=-1)) ** 2
    f = np.fft.rfftfreq(T, 1 / FS_OUT)
    feats = [np.log(P[..., (f >= lo) & (f < hi)].mean(-1) + 1e-8) for lo, hi in BANDS.values()]
    feats.append(x.var(-1))
    feats.append(np.abs(np.diff(x, axis=-1)).mean(-1))   # line length
    return np.concatenate(feats, axis=1).astype(np.float32)

def load_file_lean(path, chunk=500):
    obj = torch.load(path, weights_only=False)
    samples = obj['dataset'] if isinstance(obj, dict) and 'dataset' in obj else obj
    keys = list(samples[0].keys())
    print('sample keys:', keys)
    SUBJ = next((k for k in ('subject', 'subject_id', 'sub', 'participant') if k in keys), None)
    assert SUBJ, 'no subject key found, paste the keys printed above'
    GRAN = 'granularity' if 'granularity' in keys else None
    xs, ys, subj, gran = [], [], [], []
    for i in range(0, len(samples), chunk):
        part = samples[i:i+chunk]
        batch = np.stack([np.asarray(s['eeg_data'], dtype=np.float32) for s in part])
        xs.append(preprocess(batch)); del batch
        ys.extend(s['label'] for s in part)
        subj.extend(s[SUBJ] for s in part)
        if GRAN: gran.extend(s[GRAN] for s in part)
    del obj, samples; gc.collect()
    return (np.concatenate(xs), np.array(ys), np.array(subj),
            np.array(gran) if GRAN else None)

# only the first real file, ignoring the macOS junk files
files = [f for f in files if '__MACOSX' not in f and not os.path.basename(f).startswith('._')][:1]
print(files)

t0 = time.time()
x, y, subj, gran = load_file_lean(files[0])
print(x.shape, 'participants found:', np.unique(subj), f'{time.time()-t0:.0f}s')

for sid in np.unique(subj):
    out = f'{FEAT_DIR}/eegimagenet_S{sid}.npz'
    if os.path.exists(out):
        print('skip', out); continue
    m = subj == sid
    xs, ys = x[m], y[m]
    gs = gran[m] if gran is not None else None
    k = keep_mask(xs)
    xs, ys = xs[k], ys[k]
    X = features(zscore_channels(xs))
    extra = {'granularity': gs[k]} if gs is not None else {}
    np.savez_compressed(out, X=X, y=ys, subject=np.full(len(ys), sid), **extra)
    print(f'S{sid}: kept {k.sum()}/{len(k)} epochs, X {X.shape}')
del x; gc.collect()

In [ ]:
import glob
from sklearn.preprocessing import LabelEncoder

print(sorted(os.listdir(FEAT_DIR)))   # you should see about 8 files named eegimagenet_S...

parts = [np.load(f, allow_pickle=True) for f in sorted(glob.glob(f'{FEAT_DIR}/eegimagenet_S*.npz'))]
print(len(parts), 'participant files found')
X = np.concatenate([d['X'] for d in parts]); y = np.concatenate([d['y'] for d in parts])
subject = LabelEncoder().fit_transform(np.concatenate([d['subject'] for d in parts]))
np.savez_compressed(f'{FEAT_DIR}/eegimagenet_features.npz', X=X, y=y, subject=subject)
print(X.shape, len(set(y)), 'classes,', len(set(subject)), 'participants')

## Step 5: Baselines

First a **sanity check within one participant**. The dataset paper reports within-participant SVM accuracy of about 0.39 (all classes), 0.51 (coarse) and 0.78 (fine). Your numbers will differ, since this is a simplified pipeline, but should be well above chance.

Caution: the dataset is recorded in blocks (50 images of one category in a row), so a random split can be optimistic through slow drifts. Treat within-participant numbers as a sanity check only; **LOSO is the headline result.**

In [ ]:
import os, glob, time
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/brain_decoding'
FEAT_DIR = f'{BASE}/features'
RES_DIR = f'{BASE}/results'
os.makedirs(RES_DIR, exist_ok=True)
SEED = 0
MAX_TRAIN = 20000

print(sorted(os.listdir(FEAT_DIR)))

path = f'{FEAT_DIR}/eegimagenet_features.npz'
if not os.path.exists(path):
    # the combined file was never created, so build it from the per-participant files
    from sklearn.preprocessing import LabelEncoder
    parts = [np.load(f, allow_pickle=True) for f in sorted(glob.glob(f'{FEAT_DIR}/eegimagenet_S*.npz'))]
    print(len(parts), 'participant files found')
    X = np.concatenate([d['X'] for d in parts]); y = np.concatenate([d['y'] for d in parts])
    subject = LabelEncoder().fit_transform(np.concatenate([d['subject'] for d in parts]))
    np.savez_compressed(path, X=X, y=y, subject=subject)
else:
    d = np.load(path, allow_pickle=True)
    X, y, subject = d['X'], d['y'], d['subject']

print(X.shape, len(set(y)), 'classes,', len(set(subject)), 'participants')

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

yi = LabelEncoder().fit_transform(y)
MODELS = {
    'LinearSVC': lambda: make_pipeline(StandardScaler(), LinearSVC(C=0.01, max_iter=3000, random_state=SEED)),
    'LDA':       lambda: make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')),
    'RandomForest': lambda: RandomForestClassifier(n_estimators=100, n_jobs=2, random_state=SEED),
}
print(X.shape, len(set(y)), 'classes,', len(set(subject)), 'participants')

## Step 6: LOSO (leave one participant out) and save results

In [ ]:
# TODO: swap this loop for common/eval.py once Roshan's version is in the repo.
rng = np.random.default_rng(SEED)
rows = []
for name, mk in MODELS.items():
    for held in np.unique(subject):
        tr = np.where(subject != held)[0]; te = np.where(subject == held)[0]
        if len(tr) > MAX_TRAIN: tr = rng.choice(tr, MAX_TRAIN, replace=False)
        t0 = time.time()
        clf = mk().fit(X[tr], yi[tr]); pred = clf.predict(X[te])
        rows.append(dict(dataset='eegimagenet', task='all_classes', model=name, eval='LOSO',
                         held_out=int(held), acc=accuracy_score(yi[te], pred),
                         f1_macro=f1_score(yi[te], pred, average='macro')))
        print(f'{name:13s} held-out P{held:02d}  acc {rows[-1]["acc"]:.3f}  ({time.time()-t0:.0f}s)')
        pd.DataFrame(rows).to_csv(f'{RES_DIR}/eegimagenet.csv', index=False)   # saved after every fold
print(pd.DataFrame(rows).groupby('model')[['acc', 'f1_macro']].mean())

In [ ]:
csv_path = f'{RES_DIR}/eegimagenet.csv'
rows = pd.read_csv(csv_path).to_dict('records') if os.path.exists(csv_path) else []
done = {(r['model'], int(r['held_out'])) for r in rows}
print(len(done), 'folds already done')

for name, mk in MODELS.items():
    for held in np.unique(subject):
        if (name, int(held)) in done:
            continue
        rng = np.random.default_rng(SEED + int(held))
        tr = np.where(subject != held)[0]; te = np.where(subject == held)[0]
        if len(tr) > MAX_TRAIN: tr = rng.choice(tr, MAX_TRAIN, replace=False)
        t0 = time.time()
        clf = mk().fit(X[tr], yi[tr]); pred = clf.predict(X[te])
        rows.append(dict(dataset='eegimagenet', task='all_classes', model=name, eval='LOSO',
                         held_out=int(held), acc=accuracy_score(yi[te], pred),
                         f1_macro=f1_score(yi[te], pred, average='macro')))
        print(f'{name:13s} held-out P{held:02d}  acc {rows[-1]["acc"]:.3f}  ({time.time()-t0:.0f}s)')
        pd.DataFrame(rows).to_csv(csv_path, index=False)
print(pd.DataFrame(rows).groupby('model')[['acc', 'f1_macro']].mean())

In [ ]:
print('chance =', 1/len(set(yi)))
for s in np.unique(subject):
    m = subject == s
    c = np.bincount(yi[m], minlength=len(set(yi)))
    print(s, m.sum(), 'epochs,', (c > 0).sum(), 'classes present, per class', c.min(), '-', c.max())

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
res = []
for s in np.unique(subject):
    m = np.where(subject == s)[0]
    accs = []
    for tr, te in skf.split(X[m], yi[m]):
        clf = MODELS['LDA']().fit(X[m][tr], yi[m][tr])
        accs.append(accuracy_score(yi[m][te], clf.predict(X[m][te])))
    res.append(np.mean(accs)); print(f'participant {s}: {res[-1]:.3f}')
print('mean', np.mean(res))

In [ ]:
import glob
f = sorted(glob.glob(f'{FEAT_DIR}/eegimagenet_S*.npz'))[0]
d = np.load(f, allow_pickle=True)
print(d.files)
if 'granularity' in d.files:
    print(np.unique(d['granularity'], return_counts=True))

In [ ]:
import glob
from sklearn.model_selection import StratifiedKFold

parts = [np.load(f, allow_pickle=True) for f in sorted(glob.glob(f'{FEAT_DIR}/eegimagenet_S*.npz'))]
gran = np.concatenate([d['granularity'] for d in parts])
assert len(gran) == len(y)
for g in ['coarse', 'fine']:
    m = gran == g
    print(g, m.sum(), 'epochs,', len(set(y[m])), 'classes')

skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
tasks = {'all_classes': np.ones(len(y), bool), 'coarse': gran == 'coarse', 'fine': gran == 'fine'}
rows_w = []
for task, tm in tasks.items():
    for s in np.unique(subject):
        idx = np.where(tm & (subject == s))[0]
        yy = LabelEncoder().fit_transform(y[idx])
        accs = []
        for tr, te in skf.split(X[idx], yy):
            clf = MODELS['LDA']().fit(X[idx][tr], yy[tr])
            accs.append(accuracy_score(yy[te], clf.predict(X[idx][te])))
        rows_w.append(dict(task=task, participant=int(s), acc=np.mean(accs), chance=1 / len(set(yy))))
dfw = pd.DataFrame(rows_w)
dfw.to_csv(f'{RES_DIR}/eegimagenet_within.csv', index=False)
print(dfw.groupby('task')[['acc', 'chance']].mean())

In [ ]:
csv_path = f'{RES_DIR}/eegimagenet.csv'
rows = pd.read_csv(csv_path).to_dict('records') if os.path.exists(csv_path) else []
done = {(r['task'], r['model'], int(r['held_out'])) for r in rows}

for task in ['coarse', 'fine']:
    tm = gran == task
    for name in ['LDA', 'LinearSVC']:
        for held in np.unique(subject):
            if (task, name, int(held)) in done: continue
            rng = np.random.default_rng(SEED + int(held))
            tr = np.where(tm & (subject != held))[0]; te = np.where(tm & (subject == held))[0]
            if len(tr) > MAX_TRAIN: tr = rng.choice(tr, MAX_TRAIN, replace=False)
            le = LabelEncoder().fit(y[tm])
            clf = MODELS[name]().fit(X[tr], le.transform(y[tr]))
            pred = clf.predict(X[te]); yt = le.transform(y[te])
            rows.append(dict(dataset='eegimagenet', task=task, model=name, eval='LOSO', held_out=int(held),
                             acc=accuracy_score(yt, pred), f1_macro=f1_score(yt, pred, average='macro')))
            print(task, name, f'P{held}', f'{rows[-1]["acc"]:.3f}')
            pd.DataFrame(rows).to_csv(csv_path, index=False)
print(pd.DataFrame(rows).groupby(['task', 'model'])[['acc', 'f1_macro']].mean())

In [ ]:
dfw = pd.read_csv(f'{RES_DIR}/eegimagenet_within.csv')
print(dfw.groupby('task')[['acc', 'chance']].mean())

In [ ]:
import os, glob, time
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/brain_decoding'
FEAT_DIR = f'{BASE}/features'
RES_DIR = f'{BASE}/results'
print(sorted(os.listdir(RES_DIR)))

p = f'{RES_DIR}/eegimagenet_within.csv'
if os.path.exists(p):
    dfw = pd.read_csv(p)
    print(dfw.groupby('task')[['acc', 'chance']].mean())
else:
    print('eegimagenet_within.csv not found, so the within-participant cell (Cell 1) has not been run or saved yet')

In [ ]:
import os, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix

BASE = '/content/drive/MyDrive/brain_decoding'; FEAT_DIR = f'{BASE}/features'; RES_DIR = f'{BASE}/results'
SEED = 0

# summary table: chance vs within-participant vs LOSO (LDA)
within = pd.read_csv(f'{RES_DIR}/eegimagenet_within.csv'); loso = pd.read_csv(f'{RES_DIR}/eegimagenet.csv')
table = pd.concat([within.groupby('task')['chance'].mean().rename('chance'),
                   within.groupby('task')['acc'].mean().rename('within_participant'),
                   loso[loso.model == 'LDA'].groupby('task')['acc'].mean().rename('LOSO')], axis=1).round(3)
table.to_csv(f'{RES_DIR}/eegimagenet_summary.csv'); print(table)

# confusion matrix: participant 0, coarse task, LDA, 5-fold predictions
parts = [np.load(f, allow_pickle=True) for f in sorted(glob.glob(f'{FEAT_DIR}/eegimagenet_S*.npz'))]
X = np.concatenate([d['X'] for d in parts]); y = np.concatenate([d['y'] for d in parts])
gran = np.concatenate([d['granularity'] for d in parts])
subject = LabelEncoder().fit_transform(np.concatenate([d['subject'] for d in parts]))
idx = np.where((subject == 0) & (gran == 'coarse'))[0]
yy = LabelEncoder().fit_transform(y[idx])
clf = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))
pred = cross_val_predict(clf, X[idx], yy, cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
cm = confusion_matrix(yy, pred, normalize='true')
plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap='viridis'); plt.colorbar(label='fraction')
plt.title('EEG-ImageNet, participant 0, coarse, LDA (5-fold)'); plt.xlabel('predicted'); plt.ylabel('true')
plt.savefig(f'{RES_DIR}/eegimagenet_confusion_coarse_P0.png', dpi=150, bbox_inches='tight'); plt.show()

## Next steps (Days 3 to 4)

1. Repeat with the coarse and fine label sets once you know where the granularity labels live in the files.
2. Confusion matrix for the best model.
3. Add more participants if each takes under an hour.
4. Share `eegimagenet_features.npz` with Roshan so the learning curves can run.